# Marine Buoy Weather Forecasting
### Data Science for Business — NOAA Buoy 42002 (1980–2023)

---

Questo notebook descrive passo per passo il processo di analisi e modellazione sviluppato per il forecasting di variabili meteo-marine sulla boa NOAA **42002**, situata nel Golfo del Messico (26°N, 93°W).

L'obiettivo è costruire modelli in grado di **prevedere un'ora nel futuro** due variabili:
- `WVHT` — altezza significativa delle onde (metri)
- `WTMP` — temperatura dell'acqua (°C)

Il tutto viene poi esposto tramite un'**API REST** e un'**interfaccia web**, entrambe sviluppate con FastAPI e containerizzate con Docker.

**Dataset:** [`Qdrant/NOAA-Buoy`](https://huggingface.co/datasets/Qdrant/NOAA-Buoy) su Hugging Face  
**Repository:** https://github.com/Brusca01/marine-buoy-forecasting

**Indice:**
1. Setup
2. Caricamento del dataset (`load.py`)
3. Pulizia dei dati (`clean.py`)
4. Analisi esplorativa — EDA
5. Feature engineering e split temporale (`features.py`)
6. Modelli di regressione e confronto (`models.py`)
7. AutoML con FLAML (`automl.py`)
8. Clustering temporale (`clustering.py`)
9. Web Application (`backend/` e `frontend/`)
10. Conclusioni


## 1. Setup


In [ ]:
import sys, json, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

PROCESSED = ROOT / 'data' / 'processed'
MODELS    = ROOT / 'models'
CLUSTER   = PROCESSED / 'clustering'
RESULTS   = ROOT / 'results'
RESULTS.mkdir(exist_ok=True)

BLUE, ORANGE, GREEN = '#2563eb', '#ea580c', '#16a34a'
GRAY   = '#93c5fd'
COLORS = [BLUE, ORANGE, GREEN, '#9333ea']

print('ROOT:', ROOT)

**Cosa fa questo blocco:** importa le librerie necessarie e definisce i percorsi delle cartelle del progetto.
- `pandas` e `numpy` — manipolazione dati
- `matplotlib` — grafici
- `PROCESSED`, `MODELS`, `CLUSTER` — cartelle dove sono salvati i file generati dalla pipeline
- `RESULTS` — cartella dove vengono salvate le immagini dei grafici


---
## 2. Caricamento del dataset (`load.py`)

Il dataset [`Qdrant/NOAA-Buoy`](https://huggingface.co/datasets/Qdrant/NOAA-Buoy) è disponibile su Hugging Face e contiene misurazioni orarie della boa NOAA **42002** nel Golfo del Messico dal **1980 al 2023**.

**Problema riscontrato:** la funzione standard `load_dataset('Qdrant/NOAA-Buoy')` fallisce con `DatasetGenerationCastError` perché i file CSV mensili del repository hanno colonne inconsistenti tra loro.

**Soluzione adottata in `load.py`:** i file processati in formato `.parquet` (`full_years_remove_flawed.parquet` e `full_2023_remove_flawed.parquet`) vengono scaricati direttamente tramite `huggingface_hub`, bypassando `load_dataset`. Questi file contengono solo le righe valide già filtrate.

**Output:** `data/raw/buoy_42002.csv` con le colonne:
- Vento: `WDIR`, `WSPD`, `GST`
- Onde: `WVHT`, `DPD`, `APD`, `MWD`
- Atmosfera/mare: `PRES`, `ATMP`, `WTMP`


In [ ]:
raw_f = ROOT / 'data' / 'raw' / 'buoy_42002.csv'
if raw_f.exists():
    raw = pd.read_csv(raw_f, parse_dates=['timestamp'])
    print('Dataset grezzo (Qdrant/NOAA-Buoy):')
    print(f'  Righe:   {len(raw):,}')
    print(f'  Colonne: {list(raw.columns)}')
    print(f'  Periodo: {raw["timestamp"].min().date()} → {raw["timestamp"].max().date()}')
    display(raw.head(3))
else:
    print('File non trovato — esegui load.py prima')

**Cosa fa questo blocco:** carica il CSV grezzo salvato da `load.py` e mostra le prime righe.
Permette di verificare che il dataset sia stato scaricato correttamente e di vedere la struttura delle colonne.


---
## 3. Pulizia dei dati (`clean.py`)

`clean.py` applica due operazioni principali:

**Step 1 — Bounds fisici:** i valori fuori dai limiti fisicamente plausibili vengono sostituiti con `NaN`. Sono errori di sensore o valori sentinella usati storicamente per indicare 'dato mancante':
- `WVHT > 30m` → altezza d'onda impossibile
- `WTMP < -5°C` oppure `> 40°C` → temperatura dell'acqua irrealistica
- `PRES` fuori da 800–1100 hPa → pressione atmosferica impossibile
- analogamente per le altre variabili

**Step 2 — Resampling orario con `resample('1h').mean()`:** le misurazioni originali non sono perfettamente equispaziaste (a volte ogni 30 minuti, a volte con buchi). Il resampling crea una **griglia temporale regolare** con esattamente un'osservazione per ora.

Questo è fondamentale: se la griglia non fosse regolare, il lag 1 non significherebbe '1 ora fa' ma 'l'osservazione precedente', che potrebbe essere 30 minuti o 2 ore fa.

**Output:** `data/processed/buoy_42002_clean.csv`


In [ ]:
df = pd.read_csv(PROCESSED / 'buoy_42002_clean.csv', parse_dates=['timestamp'])
print('Dati dopo clean.py:')
print(f'  Righe:     {len(df):,}')
print(f'  Periodo:   {df["timestamp"].min().date()} → {df["timestamp"].max().date()}')
print(f'  Frequenza: oraria (griglia regolare)')
print(f'  Anni:      {df["timestamp"].dt.year.nunique()}')
print(f'\nMissing values (%):')
miss = (df.isnull().mean() * 100).round(1)
print(miss[miss > 0].to_string() if miss[miss > 0].any() else '  nessun missing')
print(f'\nStatistiche descrittive:')
display(df[['WVHT','WTMP','WSPD','PRES','ATMP']].describe().round(3))

**Cosa fa questo blocco:** carica i dati puliti e mostra le statistiche descrittive.
Verifichiamo che i bounds siano stati applicati correttamente (nessun valore anomalo rimasto), che la griglia sia oraria e che i missing values siano entro limiti accettabili.


---
## 4. Analisi Esplorativa (EDA)

Prima di costruire i modelli, analizziamo la struttura dei dati per capire:
- **Trend** — i valori cambiano nel lungo periodo?
- **Stagionalità** — ci sono pattern che si ripetono (estate/inverno)?
- **Distribuzione** — i valori sono simmetrici? Ci sono outlier?

Questa fase guida le scelte successive: la stagionalità evidenziata suggerisce di aggiungere feature di calendario; la distribuzione dei valori suggerisce se è necessaria una normalizzazione.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
df.set_index('timestamp')['WVHT'].dropna().plot(
    ax=axes[0], lw=0.3, color=BLUE, alpha=0.8)
axes[0].set_title('WVHT — Significant Wave Height (m) — Buoy 42002 (1980–2023)', fontsize=12)
axes[0].set_ylabel('m'); axes[0].grid(alpha=0.3)
df.set_index('timestamp')['WTMP'].dropna().plot(
    ax=axes[1], lw=0.3, color=ORANGE, alpha=0.8)
axes[1].set_title('WTMP — Water Temperature (°C) — Buoy 42002 (1980–2023)', fontsize=12)
axes[1].set_ylabel('°C'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS / '01_time_series.png', dpi=120, bbox_inches='tight')
plt.show()

**Cosa fa questo blocco:** visualizza le serie storiche complete di WVHT e WTMP dal 1980 al 2023.
Si nota chiaramente la **stagionalità annuale** di entrambe: WVHT con onde più alte in inverno, WTMP con temperatura più alta in estate.
Il grafico viene salvato come `results/01_time_series.png`.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6))

df['WVHT'].dropna().plot(kind='hist', bins=60, ax=axes[0][0], color=BLUE)
axes[0][0].set_title('WVHT — distribuzione'); axes[0][0].set_xlabel('m')

df['WTMP'].dropna().plot(kind='hist', bins=60, ax=axes[0][1], color=ORANGE)
axes[0][1].set_title('WTMP — distribuzione'); axes[0][1].set_xlabel('°C')

df.assign(month=df['timestamp'].dt.month).groupby('month')['WVHT'].mean().plot(
    kind='bar', ax=axes[1][0], color=BLUE)
axes[1][0].set_title('WVHT media per mese\n(onde più alte in inverno)')
axes[1][0].set_xlabel('Mese'); axes[1][0].set_ylabel('m')

df.assign(month=df['timestamp'].dt.month).groupby('month')['WTMP'].mean().plot(
    kind='bar', ax=axes[1][1], color=ORANGE)
axes[1][1].set_title('WTMP media per mese\n(acqua più calda in estate)')
axes[1][1].set_xlabel('Mese'); axes[1][1].set_ylabel('°C')

plt.suptitle('EDA — Distribuzione e Stagionalità dei Target', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / '02_eda.png', dpi=120, bbox_inches='tight')
plt.show()

**Cosa fa questo blocco:** mostra la distribuzione statistica e la stagionalità mensile dei due target.
- In alto: gli istogrammi mostrano la forma della distribuzione (WVHT è asimmetrica con coda a destra, WTMP è più simmetrica).
- In basso: i bar chart per mese confermano la stagionalità, che motiverà l'uso di feature di calendario nel modello.
Il grafico viene salvato come `results/02_eda.png`.


---
## 5. Feature Engineering e Split Temporale (`features.py`)

### Trasformazione in problema supervisionato

I modelli di machine learning lavorano su una matrice X (feature) e un vettore y (target). Per un problema di forecasting, costruiamo:
- **y** = valore del target all'ora `t+1` (quello che vogliamo predire)
- **X** = tutto ciò che sappiamo fino all'ora `t` (nessuna informazione dal futuro)

### Feature costruite

**Lag del target** — i valori passati del target sfruttano l'autocorrelazione della serie temporale:
```
WVHT_lag1  = valore 1 ora fa    ← la più predittiva
WVHT_lag2  = valore 2 ore fa
WVHT_lag3, WVHT_lag6, WVHT_lag12, WVHT_lag24
```

**Rolling statistics** — medie mobili che guardano solo nel passato:
```
WVHT_roll3  = media delle ultime 3 ore
WVHT_roll6  = media delle ultime 6 ore
WVHT_roll24 = media delle ultime 24 ore
```

**Variabili esogene** — le altre variabili meteorologiche al tempo corrente e con 1 lag:
```
WSPD_t0, PRES_t0, ATMP_t0, ...   ← valori correnti
WSPD_lag1, PRES_lag1, ...         ← valori 1 ora fa
```

**Feature di calendario con codifica ciclica** — l'ora 23 è vicina all'ora 0, ma numericamente distante. `sin` e `cos` preservano questa continuità:
```
hour_sin = sin(2π · ora / 24)    hour_cos = cos(2π · ora / 24)
doy_sin  = sin(2π · giorno / 365) doy_cos  = cos(2π · giorno / 365)
```

### Split temporale 70% / 15% / 15%

Lo split è **cronologico** — mai casuale. Uno shuffle casuale introdurrebbe data leakage: il modello vedrebbe dati futuri durante il training.

```
[========== TRAIN 70% ==========][=== VAL 15% ===][=== TEST 15% ===]
         1980 → ~2013                ~2013 → 2018       2018 → 2023
```

- **Train:** usato per addestrare i modelli
- **Validation:** usato per scegliere il modello migliore — mai per il training
- **Test:** usato UNA SOLA VOLTA per la valutazione finale — mai per decisioni

**Output:** 6 file CSV in `data/processed/` — `train_42002_WVHT.csv`, `val_42002_WVHT.csv`, `test_42002_WVHT.csv` e analoghi per WTMP.


In [ ]:
import joblib
for target in ['WVHT', 'WTMP']:
    train = pd.read_csv(PROCESSED / f'train_42002_{target}.csv', parse_dates=['timestamp'])
    val   = pd.read_csv(PROCESSED / f'val_42002_{target}.csv',   parse_dates=['timestamp'])
    test  = pd.read_csv(PROCESSED / f'test_42002_{target}.csv',  parse_dates=['timestamp'])
    feat  = [c for c in train.columns if c not in ('timestamp','y')]
    print(f'\n=== {target} — {len(feat)} feature totali ===')
    print(f'  Train: {len(train):>7,} righe [{train["timestamp"].min().date()} → {train["timestamp"].max().date()}]')
    print(f'  Val:   {len(val):>7,} righe [{val["timestamp"].min().date()} → {val["timestamp"].max().date()}]')
    print(f'  Test:  {len(test):>7,} righe [{test["timestamp"].min().date()} → {test["timestamp"].max().date()}]')
    print(f'  Prime feature: {feat[:8]}')

**Cosa fa questo blocco:** carica i 3 split per ciascun target e ne mostra dimensioni e intervalli temporali.
Si verifica che lo split sia corretto: train prima, val nel mezzo, test alla fine — nessuna sovrapposizione.
Le feature mostrate confermano la costruzione dei lag e delle feature di calendario.


---
## 6. Modelli di Regressione e Confronto (`models.py`)

Vengono addestrati e confrontati 4 approcci per ciascuno dei 2 target.

### Persistence Baseline
Prevede che il valore futuro sia uguale a quello attuale: `ŷ(t+1) = y(t)`. Non richiede training. È il riferimento minimo: qualsiasi modello ML deve superarlo per essere utile.

### Ridge Regression
Regressione lineare con regolarizzazione L2. Minimizza: `||y - Xw||² + α||w||²`. Il termine di penalità evita pesi troppo grandi e riduce l'overfitting quando le feature sono correlate (come i lag). Richiede la normalizzazione delle feature con `StandardScaler`.

### Random Forest Regressor
Costruisce molti alberi decisionali su sottocampioni casuali del dataset (**bagging**) e ne fa la media. La randomizzazione riduce la varianza rispetto a un singolo albero. Non richiede normalizzazione.

### Gradient Boosting Regressor
Costruisce alberi in sequenza (**boosting**): ogni albero viene addestrato sui **residui** del modello precedente, correggendone gli errori iterativamente. Riduce il bias progressivamente. Generalmente più accurato ma computazionalmente più pesante.

### Metriche
- **MAE** — errore medio assoluto (interpretabile: 0.1m di errore medio su WVHT)
- **RMSE** — penalizza di più gli errori grandi rispetto a MAE
- **R²** — proporzione di varianza spiegata (1.0 = perfetto, 0 = equivalente alla media)

Il modello migliore viene scelto sulla **validation RMSE**. Il test set non viene mai usato per prendere decisioni.

**Output:** bundle `models/best_WVHT.joblib` e `models/best_WTMP.joblib` contenenti modello + scaler + lista feature.


In [ ]:
metrics_f = MODELS / 'metrics.json'
if not metrics_f.exists():
    print('Esegui models.py prima')
else:
    m = json.loads(metrics_f.read_text())
    for target, info in m.items():
        print(f'\n=== {target} === Migliore su val RMSE: {info["best"].upper()}')
        rows = []
        for name, v in info['metrics'].items():
            rows.append({'model': name,
                'val MAE':v['val']['MAE'], 'val RMSE':v['val']['RMSE'], 'val R²':v['val']['R2'],
                'test MAE':v['test']['MAE'], 'test RMSE':v['test']['RMSE'], 'test R²':v['test']['R2']})
        display(pd.DataFrame(rows).sort_values('val RMSE').reset_index(drop=True))
        p = info['metrics']['persistence']['test']['RMSE']
        b = info['metrics'][info['best']]['test']['RMSE']
        print(f'  → Miglioramento su persistence: {(p-b)/p*100:.1f}%')

**Cosa fa questo blocco:** carica le metriche salvate da `models.py` e le mostra come tabella ordinata per validation RMSE.
La riga in cima è il modello migliore. Viene mostrato anche il miglioramento percentuale rispetto alla persistence baseline.


In [ ]:
if metrics_f.exists():
    m = json.loads(metrics_f.read_text())
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for row, target in enumerate(['WVHT','WTMP']):
        info = m[target]
        names = list(info['metrics'].keys())
        for col, (split, met) in enumerate([('val','RMSE'),('val','R2'),('test','RMSE')]):
            vals   = [info['metrics'][n][split][met] for n in names]
            colors = [BLUE if n==info['best'] else GRAY for n in names]
            axes[row][col].bar(names, vals, color=colors)
            axes[row][col].set_title(f'{target} — {split} {met}', fontsize=10)
            axes[row][col].tick_params(axis='x', rotation=25, labelsize=8)
            if met=='R2': axes[row][col].set_ylim(0,1)
            axes[row][col].grid(axis='y', alpha=0.3)
    plt.suptitle('Confronto modelli — blu = migliore su validation RMSE', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '03_model_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** visualizza il confronto grafico tra i modelli per entrambi i target.
- Colonna sinistra: validation RMSE (usato per la selezione)
- Colonna centrale: validation R²
- Colonna destra: test RMSE (valutazione finale)
La barra blu indica il modello selezionato. Salvato in `results/03_model_comparison.png`.


In [ ]:
if metrics_f.exists():
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    for i, target in enumerate(['WVHT','WTMP']):
        test_f   = PROCESSED / f'test_42002_{target}.csv'
        bundle_f = MODELS / f'best_{target}.joblib'
        if not test_f.exists() or not bundle_f.exists(): continue
        test   = pd.read_csv(test_f, parse_dates=['timestamp'])
        bundle = joblib.load(bundle_f)
        X = test[bundle['features']].values.astype(float)
        if bundle['scaler']: X = bundle['scaler'].transform(X)
        yhat  = bundle['model'].predict(X)
        ytrue = test['y'].values
        pers  = test[f'{target}_t0'].values
        sl = slice(-500, None)
        axes[i].plot(test['timestamp'].iloc[sl], ytrue[sl],
                     label='valore reale', lw=1.5, color='#1e293b')
        axes[i].plot(test['timestamp'].iloc[sl], yhat[sl],
                     label=f'modello ({bundle["name"]})', lw=1, color=BLUE, alpha=0.85)
        axes[i].plot(test['timestamp'].iloc[sl], pers[sl],
                     label='persistence (baseline)', lw=0.8, color=ORANGE, linestyle='--', alpha=0.7)
        axes[i].set_title(f'{target} — reale vs predetto vs persistence (ultimi 500 punti test)', fontsize=11)
        axes[i].legend(fontsize=9); axes[i].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(RESULTS / '04_predictions_vs_actual.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** applica il modello migliore agli ultimi 500 punti del test set e confronta graficamente reale, predetto e persistence.
- Nero: valore reale osservato
- Blu: previsione del modello migliore
- Arancio tratteggiato: persistence baseline
Il modello deve seguire la curva nera più da vicino della persistence. Salvato in `results/04_predictions_vs_actual.png`.


---
## 7. AutoML con FLAML (`automl.py`)

**FLAML** (Fast and Lightweight AutoML, Microsoft) è un framework di AutoML che cerca automaticamente il modello migliore e i suoi iperparametri entro un budget di tempo prefissato. Invece di configurare manualmente ogni algoritmo, FLAML esplora autonomamente combinazioni di modelli (LightGBM, XGBoost, Random Forest, Ridge e altri) e restituisce la configurazione ottimale.

**Configurazione utilizzata:**
```python
automl.fit(
    task        = 'regression',   # problema di regressione
    metric      = 'rmse',         # ottimizza l'RMSE
    time_budget = 60,             # 60 secondi di ricerca
    eval_method = 'cv',           # cross-validation
    split_type  = 'time',         # CV temporale — rispetta l'ordine cronologico
    n_splits    = 5,
)
```

`split_type='time'` garantisce che in ogni fold di cross-validation il training preceda sempre la validation, coerentemente con lo split manuale — senza data leakage.

**Output:** `models/automl_WVHT.joblib`, `models/automl_WTMP.joblib`, `models/metrics_automl.json`.


In [ ]:
automl_f = MODELS / 'metrics_automl.json'
if not automl_f.exists():
    print('Esegui automl.py prima')
else:
    a = json.loads(automl_f.read_text())
    print('Risultati AutoML — FLAML (budget 60 secondi):')
    for target, info in a.items():
        print(f'\n  {target}:')
        print(f'    Algoritmo trovato: {info["best_estimator"]}')
        print(f'    Val  → MAE={info["metrics"]["val"]["MAE"]}  RMSE={info["metrics"]["val"]["RMSE"]}  R²={info["metrics"]["val"]["R2"]}')
        print(f'    Test → MAE={info["metrics"]["test"]["MAE"]}  RMSE={info["metrics"]["test"]["RMSE"]}  R²={info["metrics"]["test"]["R2"]}')

**Cosa fa questo blocco:** carica i risultati di FLAML da `metrics_automl.json` e mostra l'algoritmo scelto automaticamente e le sue metriche.
FLAML seleziona l'algoritmo e la configurazione che minimizza l'RMSE di cross-validation temporale entro il budget di 60 secondi.


In [ ]:
if metrics_f.exists() and automl_f.exists():
    m = json.loads(metrics_f.read_text())
    a = json.loads(automl_f.read_text())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for i, target in enumerate(['WVHT','WTMP']):
        info  = m[target]
        names = list(info['metrics'].keys()) + [f'AutoML\n({a[target]["best_estimator"]})']
        test_rmse = ([info['metrics'][n]['test']['RMSE'] for n in info['metrics']]
                     + [a[target]['metrics']['test']['RMSE']])
        bar_colors = [ORANGE if 'AutoML' in n
                      else (BLUE if n.split('\n')[0].strip()==info['best'] else GRAY)
                      for n in names]
        axes[i].bar(names, test_rmse, color=bar_colors)
        axes[i].set_title(f'{target} — Test RMSE\nblu=migliore sklearn, arancio=AutoML')
        axes[i].tick_params(axis='x', rotation=20, labelsize=8)
        axes[i].grid(axis='y', alpha=0.3)
    plt.suptitle('Confronto completo — modelli classici vs AutoML (FLAML, 60s)', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '05_automl_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** confronta i modelli classici con AutoML in un unico grafico.
La barra blu è il migliore modello sklearn, la barra arancione è AutoML.
Il grafico permette di vedere se AutoML migliora ulteriormente i risultati rispetto ai modelli configurati manualmente.
Salvato in `results/05_automl_comparison.png`.


---
## 8. Clustering Temporale (`clustering.py`)

Il clustering identifica **regimi meteo-marini** ricorrenti e analizza come variano nel tempo sulla boa 42002.

Poiché il dataset contiene una sola boa, il clustering è applicato su **finestre temporali** della stessa boa invece che su boe diverse. Ogni finestra è descritta dalle sue statistiche: media e deviazione standard di `WVHT`, `WTMP`, `WSPD`, `PRES`, `ATMP` (10 feature per punto).

### Pipeline (`clustering.py`)

**1. StandardScaler** — normalizza tutte le feature su scala unitaria. KMeans calcola distanze euclidee, quindi una variabile con scala 0-1000 dominerebbe una con scala 0-1 senza normalizzazione.

**2. KMeans** — algoritmo di clustering che assegna ogni punto al centroide più vicino e aggiorna iterativamente i centroidi. Viene provato per k da 2 a 8.

**3. Silhouette Score** — per ogni k, misura la qualità dei cluster: quanto ogni punto è vicino al suo cluster rispetto al cluster più vicino. Valore 1 = perfettamente separato, 0 = sul bordo. Si sceglie il k con il punteggio più alto.

**4. Elbow Plot (Inertia)** — mostra la somma delle distanze dai centroidi al variare di k. Il 'gomito' nella curva suggerisce il k ottimale come metodo alternativo di selezione.

**5. PCA a 2 componenti** — riduce le 10 feature a 2 dimensioni per visualizzare i cluster in un grafico scatter 2D.

### Due livelli di analisi

**Clustering Statico (profili mensili):** ogni punto = un mese della boa (≈ 516 punti su 43 anni). Identifica regimi stagionali ricorrenti — es. mesi invernali con onde alte e acqua fredda vs mesi estivi con mare calmo e acqua calda.

**Clustering Dinamico (profili annuali):** ogni punto = un anno della boa (43 punti). Mostra come il regime meteorologico cambia nel tempo, evidenziando anni anomali nella storia 1980-2023.

**Output:** `data/processed/clustering/static.json`, `static.csv`, `dynamic.json`, `dynamic.csv`.


In [ ]:
static_f = CLUSTER / 'static.json'
if not static_f.exists():
    print('Esegui clustering.py prima')
else:
    s   = json.loads(static_f.read_text())
    tab = pd.read_csv(CLUSTER / 'static.csv')
    print('Clustering Statico (profili mensili):')
    print(f'  k scelto: {s["k"]}  (silhouette massima)')
    print(f'  Silhouette: {s["silhouette"]}')
    print(f'  Mesi totali: {s["n_points"]}')
    print()
    month_names = {1:'Gen',2:'Feb',3:'Mar',4:'Apr',5:'Mag',6:'Giu',
                   7:'Lug',8:'Ago',9:'Set',10:'Ott',11:'Nov',12:'Dic'}
    for c, months in s['clusters'].items():
        periods = pd.to_datetime([m+'-01' for m in months[:60]])
        top = periods.month.value_counts().sort_values(ascending=False).head(4)
        top_str = ', '.join([f'{month_names.get(mo, mo)}({v})' for mo,v in zip(top.index, top.values)])
        print(f'  Cluster {c} ({len(months)} mesi): mesi prevalenti → {top_str}')

**Cosa fa questo blocco:** mostra i risultati del clustering statico mensile.
Per ogni cluster, identifica quali mesi dell'anno sono più presenti, permettendo di caratterizzare il regime (es. Cluster 0 = mesi invernali, Cluster 1 = mesi estivi).


In [ ]:
if static_f.exists():
    s   = json.loads(static_f.read_text())
    tab = pd.read_csv(CLUSTER / 'static.csv')
    fig = plt.figure(figsize=(15, 5))
    gs  = gridspec.GridSpec(1, 3, figure=fig)

    ax0 = fig.add_subplot(gs[0])
    for c in sorted(tab['cluster'].unique()):
        sub = tab[tab['cluster']==c]
        ax0.scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}',
                    s=15, color=COLORS[c % len(COLORS)], alpha=0.7)
    ax0.set_xlabel('PCA 1'); ax0.set_ylabel('PCA 2')
    ax0.set_title(f'PCA 2D — ogni punto = 1 mese\nk={s["k"]} sil={s["silhouette"]}')
    ax0.legend(); ax0.grid(alpha=0.3)

    ax1 = fig.add_subplot(gs[1])
    sil_k = sorted([int(k) for k in s['silhouette_by_k']])
    sil_v = [s['silhouette_by_k'][str(k)] for k in sil_k]
    ax1.bar([f'k={k}' for k in sil_k], sil_v,
            color=[BLUE if k==s['k'] else GRAY for k in sil_k])
    ax1.set_title('Silhouette Score per k\npiù alto = cluster più separati\nblu = k scelto')
    ax1.set_ylim(0, max(sil_v)*1.2 if sil_v else 1); ax1.grid(axis='y', alpha=0.3)

    ax2 = fig.add_subplot(gs[2])
    inertia_k = sorted([int(k) for k in s['inertia_by_k']])
    inertia_v = [s['inertia_by_k'][str(k)] for k in inertia_k]
    ax2.plot([f'k={k}' for k in inertia_k], inertia_v, 'o-', color=BLUE, lw=2, markersize=8)
    ax2.set_title('Elbow Plot — Inertia per k\ncerca il gomito nella curva')
    ax2.grid(alpha=0.3)

    plt.suptitle('Clustering Statico — Profili Mensili (516 punti ≈ 43 anni × 12 mesi)', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '06_static_clustering.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** visualizza i tre elementi chiave del clustering statico:
- **Scatter PCA 2D** (sinistra): ogni punto è un mese, colorato per cluster. I cluster ben separati confermano che i regimi sono distinti.
- **Silhouette per k** (centro): bar chart con il punteggio per ogni k testato. La barra blu è il k scelto automaticamente.
- **Elbow Plot** (destra): mostra l'inertia al variare di k. Il gomito conferma visivamente la scelta.
Salvato in `results/06_static_clustering.png`.


In [ ]:
dyn_f = CLUSTER / 'dynamic.json'
if not dyn_f.exists():
    print('Esegui clustering.py prima')
else:
    d    = json.loads(dyn_f.read_text())
    dtab = pd.read_csv(CLUSTER / 'dynamic.csv')
    seq  = list(d['sequences'].values())[0]
    print('Clustering Dinamico (profili annuali):')
    print(f'  k={d["k"]}  silhouette={d["silhouette"]}  anni={d["n_points"]}')
    print(f'  Sequenza regime: {" → ".join(map(str, seq["clusters"]))}')
    print(f'  Cambi di regime: {seq["changes"]}')
    if seq['changes'] > 0:
        for j in range(len(seq['clusters'])-1):
            if seq['clusters'][j] != seq['clusters'][j+1]:
                print(f'    Anno {seq["years"][j+1]}: cluster {seq["clusters"][j]} → {seq["clusters"][j+1]}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for c in sorted(dtab['cluster'].unique()):
        sub = dtab[dtab['cluster']==c]
        axes[0].scatter(sub['pca_x'], sub['pca_y'], label=f'Cluster {c}',
                        color=COLORS[c % len(COLORS)], s=80, zorder=3)
        for _, r in sub.iterrows():
            axes[0].annotate(str(int(r['year'])), (r['pca_x'], r['pca_y']),
                             fontsize=7, xytext=(4,4), textcoords='offset points')
    axes[0].set_xlabel('PCA 1'); axes[0].set_ylabel('PCA 2')
    axes[0].set_title(f'PCA 2D — ogni punto = 1 anno\nk={d["k"]} sil={d["silhouette"]}')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    for yr, cl in zip(seq['years'], seq['clusters']):
        axes[1].bar(yr, 1, color=COLORS[cl % len(COLORS)], width=0.8)
    axes[1].set_title('Regime per anno — Buoy 42002 (1980–2023)\ncolore = cluster di appartenenza')
    axes[1].set_xlabel('Anno'); axes[1].set_yticks([])
    handles = [mpatches.Patch(color=COLORS[c], label=f'Cluster {c}')
               for c in sorted(set(seq['clusters']))]
    axes[1].legend(handles=handles, loc='upper right')
    axes[1].grid(axis='x', alpha=0.2)

    plt.suptitle('Clustering Dinamico — Come i regimi variano anno per anno (1980–2023)', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / '07_dynamic_clustering.png', dpi=120, bbox_inches='tight')
    plt.show()

**Cosa fa questo blocco:** visualizza il clustering dinamico su profili annuali.
- **Scatter PCA 2D** (sinistra): ogni punto è un anno, con l'etichetta dell'anno. Anni vicini nello scatter hanno condizioni meteorologiche simili.
- **Bar chart regime** (destra): mostra il cluster di appartenenza per ogni anno dal 1980 al 2023. Un cambio di colore indica un anno in cui la boa è passata a un regime diverso.
Salvato in `results/07_dynamic_clustering.png`.


---
## 9. Web Application (`backend/` e `frontend/`)

Il modello viene esposto tramite un'applicazione web composta da due componenti FastAPI:

### `API_App/backend/main.py` — REST API (porta 8080)
Carica i bundle `.joblib` salvati durante il training, esegue l'inferenza all'arrivo di una richiesta e restituisce i risultati in formato JSON. Usa **Pydantic** per la validazione automatica dei dati di input e output. I contratti Pydantic (schemi request/response) sono definiti in `backend/schemas.py`.

| Endpoint | Descrizione |
|---|---|
| `GET /api/health` | Stato del servizio |
| `GET /api/predict` | Previsione WVHT + WTMP a t+1h |
| `GET /api/comparison` | MAE/RMSE/R² di tutti i modelli |
| `GET /api/clusters/static` | Risultati clustering mensile |
| `GET /api/clusters/dynamic` | Risultati clustering annuale |

### `API_App/frontend/main.py` — Web Interface (porta 8000)
Chiama il backend tramite HTTP interno con `httpx`, riceve i dati JSON e li renderizza in pagine HTML usando **Jinja2** come template engine. I grafici interattivi nelle pagine sono generati lato browser da **Chart.js** (caricato via CDN).

I file HTML (`base.html`, `index.html`, `prediction.html`, `comparison.html`, `clustering.html`) e il CSS (`static/style.css`) definiscono l'interfaccia grafica dell'applicazione e sono stati generati nel corso del progetto.

### `API_App/Dockerfile` e `API_App/supervisord.conf`
Un singolo `Dockerfile` costruisce l'immagine partendo da `python:3.11-slim`. **Supervisord** è un process manager che avvia e monitora entrambi i processi (backend e frontend) all'interno dello stesso container, riavviandoli automaticamente in caso di crash.

### Come avviare il progetto

Tutti i comandi vanno eseguiti dalla cartella root del progetto (quella che contiene `src/` e `API_App/`).

**Build dell'immagine (una volta sola, o dopo modifiche al codice):**
```bash
docker build -f API_App/Dockerfile -t marine-buoy .
```

**Pipeline completa (scarica dati + addestra modelli):**

*Linux / macOS:*
```bash
docker run --rm \
  -v "$PWD/data:/app/data" \
  -v "$PWD/models:/app/models" \
  marine-buoy bash -c "
    python src/load.py &&
    python src/clean.py &&
    python src/features.py &&
    python src/models.py &&
    python src/automl.py 60 &&
    python src/clustering.py"
```

*Windows (PowerShell):*
```powershell
docker run --rm -v "C:\percorso\al\progetto\data:/app/data" -v "C:\percorso\al\progetto\models:/app/models" marine-buoy bash -c "python src/load.py && python src/clean.py && python src/features.py && python src/models.py && python src/automl.py 60 && python src/clustering.py"
```

**Avvio dell'applicazione:**

*Linux / macOS:*
```bash
docker run --rm -p 8000:8000 -p 8080:8080 \
  -v "$PWD/data:/app/data" \
  -v "$PWD/models:/app/models" \
  marine-buoy
```

*Windows (PowerShell):*
```powershell
docker run --rm -p 8000:8000 -p 8080:8080 -v "C:\percorso\al\progetto\data:/app/data" -v "C:\percorso\al\progetto\models:/app/models" marine-buoy
```

- Frontend → **http://localhost:8000**
- Backend API → **http://localhost:8080/docs**


---
## 10. Conclusioni

### Pipeline sviluppata

| File | Descrizione |
|---|---|
| `src/load.py` | Scarica i dati da `Qdrant/NOAA-Buoy` tramite `huggingface_hub` (bypass del bug di `load_dataset`) |
| `src/clean.py` | Applica bounds fisici e resampling orario per una griglia temporale regolare |
| `src/features.py` | Costruisce lag, rolling, feature di calendario; split temporale 70/15/15 |
| `src/models.py` | Addestra persistence, Ridge, RF, GradientBoosting; salva il migliore su val RMSE |
| `src/automl.py` | FLAML AutoML con CV temporale (`split_type='time'`); confronto con i modelli classici |
| `src/clustering.py` | KMeans su profili mensili (statico) e annuali (dinamico); silhouette, elbow, PCA |
| `src/predictor.py` | Carica i bundle, esegue inferenza, legge i risultati di metriche e clustering |
| `API_App/backend/` | FastAPI REST API — espone i risultati in JSON |
| `API_App/frontend/` | FastAPI HTML — interfaccia web con Jinja2 e Chart.js |
| `API_App/Dockerfile` | Container Docker con supervisord per avviare backend e frontend |

### Risultati principali

- **WVHT** è ben predicibile grazie all'alta autocorrelazione a breve termine: il valore dell'ora precedente è già molto informativo, e i modelli ML migliorano ulteriormente catturando pattern non lineari nelle variabili esogene.
- **WTMP** mostra forte stagionalità annuale; i lag orari catturano bene le variazioni di breve periodo.
- La **persistence baseline** è un riferimento impegnativo su serie ad alta autocorrelazione, ma i modelli ML riescono a migliorarla.
- **AutoML (FLAML)** trova automaticamente algoritmi (tipicamente LightGBM o XGBoost) che spesso superano i modelli configurati manualmente.
- Il **clustering statico** conferma la stagionalità identificata nell'EDA: i cluster separano naturalmente regimi invernali e estivi.
- Il **clustering dinamico** permette di identificare anni anomali e osservare come le condizioni meteorologiche della boa siano cambiate dal 1980 al 2023.

---
**Repository:** https://github.com/Brusca01/marine-buoy-forecasting  
**Dataset:** https://huggingface.co/datasets/Qdrant/NOAA-Buoy  
